In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

def parse_date(col_name):
    return F.coalesce(
        F.try_to_date(F.col(col_name), "yyyy/MM/dd HH:mm"),
        F.try_to_date(F.col(col_name), "yyyy-MM-dd'T'HH:mm:ss"),
        F.try_to_date(F.col(col_name), "yyyy-MM-dd HH:mm:ss"),
        F.try_to_date(F.col(col_name), "yyyy-MM-dd HH:mm:ss"),
        F.try_to_date(F.col(col_name), "yyyy-MM-dd"),
        F.try_to_date(F.col(col_name), "yyyy/MM/dd"),
        F.try_to_date(F.col(col_name), "dd/MM/yyyy"),
        F.try_to_date(F.col(col_name), "dd/MM/yyyy HH:mm")
    )

In [0]:
df_atendimento = spark.table("workspace.bronze.tb_atendimentos")

df_atendimento = df_atendimento.select(
    parse_date("created_at").alias("created_at"),
    F.col("customer_code"),
    F.initcap(F.col("event_type")).alias("event_type"),
    F.initcap(F.col("order_id")).alias("order_id"),
    F.initcap(F.col("severity")).alias("severity"),
    F.initcap(F.col("status")).alias("status"),
    F.initcap(F.col("ticket_id")).alias("ticket_id"),
    F.col("metadata")
)

df_atendimento.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_atendimentos")

In [0]:
df_entrega = spark.table("workspace.bronze.tb_entrega")

state_col = F.lower(F.trim(F.col("state")))
state_expr = (
    F.when(state_col.isin("ac", "acre"), "AC")
     .when(state_col.isin("al", "alagoas"), "AL")
     .when(state_col.isin("ap", "amapá", "amapa"), "AP")
     .when(state_col.isin("am", "amazonas"), "AM")
     .when(state_col.isin("ba", "bahia"), "BA")
     .when(state_col.isin("ce", "ceará", "ceara"), "CE")
     .when(state_col.isin("df", "distrito federal"), "DF")
     .when(state_col.isin("es", "espírito santo", "espirito santo"), "ES")
     .when(state_col.isin("go", "goiás", "goias"), "GO")
     .when(state_col.isin("ma", "maranhão", "maranhao"), "MA")
     .when(state_col.isin("mt", "mato grosso"), "MT")
     .when(state_col.isin("ms", "mato grosso do sul"), "MS")
     .when(state_col.isin("mg", "minas gerais"), "MG")
     .when(state_col.isin("pa", "pará", "para"), "PA")
     .when(state_col.isin("pb", "paraíba", "paraiba"), "PB")
     .when(state_col.isin("pr", "paraná", "parana"), "PR")
     .when(state_col.isin("pe", "pernambuco"), "PE")
     .when(state_col.isin("pi", "piauí", "piaui"), "PI")
     .when(state_col.isin("rj", "rio de janeiro"), "RJ")
     .when(state_col.isin("rn", "rio grande do norte"), "RN")
     .when(state_col.isin("rs", "rio grande do sul"), "RS")
     .when(state_col.isin("ro", "rondônia", "rondonia"), "RO")
     .when(state_col.isin("rr", "roraima"), "RR")
     .when(state_col.isin("sc", "santa catarina", "sta catarina", "s. catarina"), "SC")
     .when(state_col.isin("sp", "são paulo", "sao paulo"), "SP")
     .when(state_col.isin("se", "sergipe"), "SE")
     .when(state_col.isin("to", "tocantins"), "TO")
     .otherwise(None)
)

df_entrega = df_entrega.select(
    F.initcap(F.col("delivery_id")).alias("delivery_id"),
    F.initcap(F.col("order_ref")).alias("order_ref"),
    F.initcap(F.col("carrier_name")).alias("carrier_name"),
    F.initcap(F.col("carrier_mode")).alias("carrier_mode"),
    F.initcap(F.col("delivery_status")).alias("delivery_status"),
    parse_date("shipped_at").alias("shipped_at"),
    parse_date("delivered_at").alias("delivered_at"),
    state_expr.alias("state"),
    F.initcap(F.col("city")).alias("city"),
    F.when(
        F.lower(F.trim(F.col("cost").cast("string"))).isin("unknown", "") | F.col("cost").isNull(), None
    ).otherwise(
        F.regexp_replace(F.col("cost").cast("string"), ",", ".").cast(DecimalType(10, 2))
    ).alias("cost")
)

df_entrega.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.tb_entrega")

In [0]:
df_clientes = spark.table("workspace.bronze.dim_clientes")

state_col = F.lower(F.trim(F.col("estado")))
state_expr = (
    F.when(state_col.isin("ac", "acre"), "AC")
     .when(state_col.isin("al", "alagoas"), "AL")
     .when(state_col.isin("ap", "amapá", "amapa"), "AP")
     .when(state_col.isin("am", "amazonas"), "AM")
     .when(state_col.isin("ba", "bahia"), "BA")
     .when(state_col.isin("ce", "ceará", "ceara"), "CE")
     .when(state_col.isin("df", "distrito federal"), "DF")
     .when(state_col.isin("es", "espírito santo", "espirito santo"), "ES")
     .when(state_col.isin("go", "goiás", "goias"), "GO")
     .when(state_col.isin("ma", "maranhão", "maranhao"), "MA")
     .when(state_col.isin("mt", "mato grosso"), "MT")
     .when(state_col.isin("ms", "mato grosso do sul"), "MS")
     .when(state_col.isin("mg", "minas gerais"), "MG")
     .when(state_col.isin("pa", "pará", "para"), "PA")
     .when(state_col.isin("pb", "paraíba", "paraiba"), "PB")
     .when(state_col.isin("pr", "paraná", "parana"), "PR")
     .when(state_col.isin("pe", "pernambuco"), "PE")
     .when(state_col.isin("pi", "piauí", "piaui"), "PI")
     .when(state_col.isin("rj", "rio de janeiro"), "RJ")
     .when(state_col.isin("rn", "rio grande do norte"), "RN")
     .when(state_col.isin("rs", "rio grande do sul"), "RS")
     .when(state_col.isin("ro", "rondônia", "rondonia"), "RO")
     .when(state_col.isin("rr", "roraima"), "RR")
     .when(state_col.isin("sc", "santa catarina", "sta catarina", "s. catarina"), "SC")
     .when(state_col.isin("sp", "são paulo", "sao paulo"), "SP")
     .when(state_col.isin("se", "sergipe"), "SE")
     .when(state_col.isin("to", "tocantins"), "TO")
     .otherwise(None)
)

df_clientes = df_clientes.select(
    F.initcap(F.col("customer_id")).alias("customer_id"),
    F.initcap(F.trim(F.col("nome_cliente"))).alias("nome_cliente"),
    F.when(F.col("segmento").isNotNull(), F.initcap(F.trim(F.col("segmento")))).otherwise(None).alias("segmento"),
    F.when(F.col("porte").isNotNull(), F.initcap(F.trim(F.col("porte")))).otherwise(None).alias("porte"),
    F.initcap(F.trim(F.col("cidade"))).alias("cidade"),
    state_expr.alias("estado"),
    F.when(F.col("status_cliente").isNotNull(), F.initcap(F.trim(F.col("status_cliente")))).otherwise(None).alias("status_cliente"),
    parse_date("data_cadastro").alias("data_cadastro"),
    F.col("email"),
    F.to_timestamp(F.col("updated_at"), "yyyy-MM-dd HH:mm:ss").alias("updated_at")
)

df_clientes.write \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.dim_clientes")

In [0]:
df_canais = spark.table("workspace.bronze.dim_canais")

df_canais = df_canais.select(
    F.upper(F.col("id_canal")).alias("id_canal"),
    F.initcap(F.col("nome_canal")).alias("nome_canal"),
    F.initcap(F.col("tipo_canal")).alias("tipo_canal"),
    F.initcap(F.col("ativo")).alias("ativo"),
    F.initcap(F.col("observacao")).alias("observacao")
)

df_canais.filter("not (id_canal = 'CH05' and observacao is not null)") \
         .write \
         .mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable("workspace.silver.dim_canais")

In [0]:
df_pedidos_itens = spark.table("workspace.bronze.tb_pedidos_itens")

df_pedidos_itens = df_pedidos_itens.select(
    F.upper(F.col("order_id")).alias("order_id"),
    F.col("item_seq"),
    F.upper(F.col("product_code")).alias("product_code"),
    F.col("quantity"),
    F.regexp_replace(F.col("unit_price"), ",", ".").cast("double").alias("unit_price"),
    F.col("total_item"),
    F.when(F.col("item_status").isNull(), None)
     .otherwise(F.initcap(F.lower(F.col("item_status"))))
     .alias("item_status")
)

df_pedidos_itens.write.mode("overwrite").saveAsTable("workspace.silver.tb_pedidos_itens")

In [0]:
df_pedidos_cabecalho = spark.table("workspace.bronze.tb_pedidos_cabecalho")
df_pedidos_cabecalho = df_pedidos_cabecalho.select(
    F.initcap(F.col("order_id")).alias("order_id"),
    F.initcap(F.col("customer_code")).alias("customer_code"),
    F.initcap(F.col("seller_id")).alias("seller_id"),
    F.initcap(F.col("status_order")).alias("status_order"),
    parse_date("order_date").alias("order_date"),
    parse_date("promised_date").alias("promised_date"),
    F.regexp_replace(F.col("gross_amount"), ",", ".").try_cast("double").alias("gross_amount_num"),
    F.col("discount_amount"),
    F.col("net_amount"),
    F.col("payment_details"),
    F.col("last_update")
)

df_pedidos_cabecalho.write.mode("overwrite").saveAsTable("workspace.silver.tb_pedidos_cabecalho")
  

In [0]:


df_produto = spark.table("workspace.bronze.dim_produto")

df_produto = df_produto.select(
    F.initcap(F.col("product_id")).alias("product_id"),
    F.initcap(F.col("name")).alias("name"),
    F.initcap(F.col("category")).alias("category"),
    F.initcap(F.col("subcategory")).alias("subcategory"),
    F.when(F.col("status").isNotNull(), F.initcap(F.col("status"))).otherwise(None).alias("status"),
    F.when(F.col("family").isNotNull(), F.initcap(F.col("family"))).otherwise(None).alias("family"),
    F.regexp_replace(F.col("list_price"), ",", ".").try_cast("double").alias("list_price"),
    F.col("currency"),
    F.col("tags"),
    F.to_timestamp(F.col("updated_at")).alias("updated_at")
)

df_produto.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_produto")

In [0]:
df_vendedores = spark.table("workspace.bronze.dim_vendedores")
df_vendedores = df_vendedores.select(
    F.initcap(F.col("seller_id")).alias("seller_id"),
    F.initcap(F.col("seller_name")).alias("seller_name"),
    F.when(F.col("canal_id").isNotNull(), F.upper(F.col("canal_id"))).otherwise(None).alias("canal_id"),
    F.when(F.col("regional_code")=='sul', 'S').otherwise(F.col("regional_code")).alias("regional_code"),
    parse_date("hire_date").alias("hire_date"),
    F.when(F.col("status").isNotNull(), F.initcap(F.col("status"))).otherwise(None).alias("status")
)

df_vendedores.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_vendedores")
    

In [0]:
df_regioes = spark.table("workspace.bronze.dim_regioes")

df_regioes \
    .filter("regional_code not in ('sul','XX')") \
    .filter("state <> 'sao paulo'") \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_regioes")